# Lesson 00 — Building and Pushing the GPU Docker Image

This notebook walks you through building the course Docker image and pushing it to Amazon ECR.

**You only need to do this once.** Every other lesson uses the image you push here.

By the end you'll understand:
- Why we need Docker for GPU workloads
- What's inside the `Dockerfile`
- How ECR authentication works
- How to verify the push succeeded

## Step 1 — Load configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# .env lives two directories up from this notebook (repo root)
repo_root = Path("../..")
load_dotenv(repo_root / ".env")

region  = os.environ.get("AWS_DEFAULT_REGION", "ap-northeast-1")
ecr_uri = os.environ.get("ECR_IMAGE_URI", "(not set yet — we'll build it below)")

print(f"Region  : {region}")
print(f"ECR URI : {ecr_uri}")

## Step 2 — Inspect the Dockerfile

Let's read the `Dockerfile` and explain what each section does.

In [ ]:
dockerfile_path = repo_root / "Dockerfile"

with open(dockerfile_path) as f:
    contents = f.read()

print(contents)

**Three things to notice:**

| Line | Why it matters |
|------|----------------|
| `pytorch/pytorch:2.1.0-cuda11.8-cudnn8-runtime` | The base image includes Python + PyTorch + CUDA 11.8. The g4dn.xlarge runs a driver that supports CUDA 11.x — this version is intentional. |
| `RUN pip install ...` | All dependencies in one layer. Docker caches this — if you don't change the pip list, subsequent builds skip this step entirely. |
| `COPY lessons/ /app/lessons/` | Every lesson script is baked in. Batch overrides `command` at submit time to run the right script. |

## Step 3 — Get your AWS account ID

ECR image URIs always include your account ID. Let's fetch it once.

In [ ]:
import boto3

sts = boto3.client("sts", region_name=region)
account_id = sts.get_caller_identity()["Account"]

ecr_host    = f"{account_id}.dkr.ecr.{region}.amazonaws.com"
image_uri   = f"{ecr_host}/gpu-teaching:latest"

print(f"Account ID : {account_id}")
print(f"ECR host   : {ecr_host}")
print(f"Image URI  : {image_uri}")
print()
print("Add this to your .env:")
print(f"ECR_IMAGE_URI={image_uri}")

## Step 4 — Create the ECR repository

This only needs to be done once. If the repository already exists, AWS returns an error — that's fine, we catch it.

In [ ]:
from botocore.exceptions import ClientError

ecr = boto3.client("ecr", region_name=region)

try:
    resp = ecr.create_repository(repositoryName="gpu-teaching")
    print(f"✅ Created ECR repository: {resp['repository']['repositoryUri']}")
except ClientError as e:
    if e.response["Error"]["Code"] == "RepositoryAlreadyExistsException":
        print("ℹ️  Repository already exists — that's fine, continuing.")
    else:
        raise

## Step 5 — Authenticate Docker with ECR

ECR uses temporary tokens (valid 12 hours). We fetch one via the SDK and pipe it to `docker login`.

In [ ]:
# Fetch an ECR auth token and pipe it straight to docker login
!aws ecr get-login-password --region {region} \
    | docker login --username AWS --password-stdin {ecr_host}

You should see `Login Succeeded`. This token is valid for 12 hours.

## Step 6 — Build the image

**This takes 3–8 minutes the first time** (downloading the ~3 GB PyTorch base image + installing packages).  
Subsequent builds are fast because Docker caches each unchanged layer.

In [ ]:
# Build from the repo root (where the Dockerfile lives)
!docker build -t gpu-teaching {repo_root}

## Step 7 — Tag and push to ECR

In [ ]:
!docker tag gpu-teaching:latest {image_uri}
!docker push {image_uri}

## Step 8 — Verify the image is in ECR

We'll use `boto3` to confirm the image was pushed successfully and check its size.

In [ ]:
resp = ecr.describe_images(
    repositoryName="gpu-teaching",
    imageIds=[{"imageTag": "latest"}]
)

image = resp["imageDetails"][0]
size_mb = image["imageSizeInBytes"] / 1_000_000

print(f"✅ Image verified in ECR")
print(f"   Repository : gpu-teaching")
print(f"   Tag        : {image['imageTags'][0]}")
print(f"   Pushed at  : {image['imagePushedAt']}")
print(f"   Size       : {size_mb:.0f} MB")
print()
print(f"Update your .env with:")
print(f"ECR_IMAGE_URI={image_uri}")

## (Optional) Step 9 — Test the image locally

Verify all Python imports work before deploying to the cloud.

In [ ]:
# Run without GPU — tests that the image starts and imports all packages correctly
!docker run --rm gpu-teaching python -c \
    "import torch, clip, cv2, boto3; print('All imports OK'); print('PyTorch:', torch.__version__)"

---

## What you just built

```
Local machine                         Amazon ECR
─────────────────────────────────     ──────────────────────────────
Dockerfile + lessons/           →     gpu-teaching:latest
  (pytorch + clip + opencv + …)       (stored, ready for Batch)
```

When you submit a Batch job in the next lessons, AWS pulls this image onto a `g4dn.xlarge`, runs your script, and shuts down the instance — all automatically.

> **Key takeaway**: The Docker image is the **unit of deployment** for all GPU work in this course.  
> Build it once, push it to ECR, and every lesson uses it — you never touch the image again unless you add a new Python dependency.

---

## Next lesson → [01 — Why GPU?](../01-why-gpu/notebook.ipynb)

We'll prove why GPUs are so much faster than CPUs for matrix math — the foundation of everything else in this course.